# 03 Portfolio Backtest

Notebook này chạy nhiều symbol rồi gộp thành portfolio equal-weight.

Điểm mới:
- Dashboard portfolio gồm combined equity, drawdown, per-symbol equity và đóng góp PnL.
- Bảng per-symbol contribution để biết symbol nào kéo lên/kéo xuống.
- Trade explorer tổng hợp toàn portfolio.


In [ ]:
# Cell 1 - Bootstrap đường dẫn import an toàn
#
# Notebook có thể được mở từ repo root, từ thư mục strategies/combo,
# hoặc từ một working directory khác trong VS Code/Jupyter. Vì vậy ta không
# dùng `config.py` làm marker root: trong strategies/combo cũng có config.py,
# rất dễ nhận nhầm thư mục strategy là repo root.
#
# Marker đáng tin cậy hơn là pyproject.toml + thư mục core_python/shared.
# Sau khi tìm được root thật, ta thêm cả repo root và core_python vào sys.path
# để import được `shared.*` và `strategies.combo.*`.

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Không tìm thấy repo root chứa {marker!r} và core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Import thư viện và helper portfolio dashboard

from IPython.display import display

from core_python.strategies.combo.params import SYMBOLS, summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    configure_notebook,
    export_result_bundle,
    plot_portfolio_dashboard,
    show_ftmo_check,
    show_note,
    show_portfolio_summary,
    show_run_config,
    show_trade_explorer,
)
from core_python.strategies.combo.portfolio.backtest import run_portfolio_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
# Cell 3 - Cấu hình portfolio backtest
#
# `symbols`: danh sách symbol sẽ chạy.
# Engine hiện chia đều vốn ban đầu cho từng symbol sleeve.
# `symbol_overrides`: override riêng từng symbol, dạng {'US30': {'x': 12.0}, ...}.

RUN_CONFIG = {
    'symbols': ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD'],
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 30_000,
    'indicator_overrides': {},
    'symbol_overrides': {},
    'export_report': False,
}

show_run_config('Cấu hình portfolio backtest', RUN_CONFIG)


In [ ]:
# Cell 4 - Chạy portfolio backtest
#
# Pipeline:
# 1. Chia vốn đều cho từng symbol.
# 2. Chạy run_symbol_backtest() cho từng symbol.
# 3. Gom trade log.
# 4. Align equity theo thời gian.
# 5. Tính combined equity và metrics cấp portfolio.

portfolio = run_portfolio_backtest(
    symbol_keys=RUN_CONFIG['symbols'],
    initial_balance=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    indicator_overrides=RUN_CONFIG['indicator_overrides'] or None,
    symbol_overrides=RUN_CONFIG['symbol_overrides'] or None,
    max_bars=RUN_CONFIG['max_bars'],
)

print('Mode              =', portfolio.account_mode)
print('Symbols           =', portfolio.symbol_keys)
print('Portfolio trades  =', len(portfolio.trades))
print('Equity rows       =', len(portfolio.combined_equity))
if len(portfolio.combined_equity):
    print('Equity range      =', portfolio.combined_equity.index.min(), '->', portfolio.combined_equity.index.max())
    print('Final equity      =', round(float(portfolio.combined_equity.iloc[-1]), 2))


In [ ]:
# Cell 5 - KPI portfolio và per-symbol contribution
#
# Đọc bảng per-symbol để biết portfolio đang phụ thuộc quá nhiều vào symbol nào,
# hoặc symbol nào có drawdown/trade count bất thường.

per_symbol_df = show_portfolio_summary(portfolio)
show_ftmo_check(portfolio.metrics)


In [ ]:
# Cell 6 - Dashboard portfolio trực quan
#
# Combined equity cho biết portfolio tổng thể.
# Per-symbol equity và PnL contribution cho biết cấu phần nào đang đóng góp hoặc gây hại.

plot_portfolio_dashboard(portfolio)


In [ ]:
# Cell 7 - Trade explorer toàn portfolio
#
# Dùng để soi chuỗi lệnh tổng hợp: symbol nào vào lệnh nhiều, lý do thoát lệnh nào phổ biến,
# và top winners/losers thuộc symbol nào.

portfolio_trades_df = show_trade_explorer(portfolio.trades, tail=60, title='Portfolio trade explorer')


In [ ]:
# Cell 8 - Export nếu cần

if RUN_CONFIG.get('export_report'):
    export_result_bundle(
        f"portfolio_{RUN_CONFIG['account_mode']}_backtest",
        metrics=portfolio.metrics,
        trades=portfolio.trades,
        equity=portfolio.combined_equity,
    )
else:
    print("Export đang tắt. Đổi RUN_CONFIG['export_report'] = True nếu muốn lưu CSV.")
